# Modelo Mono-modal: DSTFS Adaptado para Tarea Única

Este cuaderno implementa el entrenamiento y la evaluación del **DSTFS-adapted** (Deep Soft Threshold Feature Separation) adaptado exclusivamente para la tarea de regresión temporal en la disipación de huellas térmicas sobre superficies.

Incorpora el mecanismo de activación de **Umbral Suave PReLU (SPRelu)** para mitigar el ruido débil e inestable residual, y utiliza la función de pérdida robusta **SqrtScaledMSELoss** para amplificar el gradiente en las etapas iniciales de decaimiento rápido.

In [1]:
import sys
import random
from pathlib import Path

# Añadir el directorio raíz al path de Python para permitir importaciones correctas
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

from src.models.dstfs import ThermalDepartureTimeNet
from src.loaders.dstfs_loader import ThermalTraceDataset
from src.utils import SqrtScaledMSELoss, eval_dstfs_metrics, split_by_sequence

### 1. Semilla Global y Reproducibilidad

In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

### 2. Configuración General (Hiperparámetros)

In [3]:
CONFIG = {
    "epochs": 120,
    "patience": 15,
    "min_delta": 1.0,
    "batch_size": 16,
    "lr": 0.0005,          # SGD lr inicial de 0.0005
    "weight_decay": 0.0005, # DSTFS: Regularización L2 de 0.0005
    "momentum": 0.9,       # SGD Momentum de 0.9
    "min_time_s": 0.0,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "time_scale": 30.0,
}

### 3. Instanciación del Dataset y Dataloaders

In [4]:
dev = CONFIG["device"]

# Los datasets esperan las rutas relativas al directorio principal
train_ds_full = ThermalTraceDataset(metadata_csv="../processed_data/metadata_train.csv", is_train=True, min_time_s=CONFIG["min_time_s"])
val_ds_full = ThermalTraceDataset(metadata_csv="../processed_data/metadata_train.csv", is_train=False, min_time_s=CONFIG["min_time_s"])

# Adaptamos la ruta raíz de los datasets para que apunte al directorio correcto en notebooks
train_ds_full.root = Path("../processed_data")
val_ds_full.root = Path("../processed_data")

t_idx, v_idx = split_by_sequence(train_ds_full.df)

train_loader = DataLoader(Subset(train_ds_full, t_idx), batch_size=CONFIG["batch_size"], shuffle=True, pin_memory=True)
val_loader = DataLoader(Subset(val_ds_full, v_idx), batch_size=CONFIG["batch_size"], pin_memory=True)
train_eval_loader = DataLoader(Subset(train_ds_full, t_idx), batch_size=CONFIG["batch_size"], pin_memory=True)

print(f"Datos cargados: {len(t_idx)} train / {len(v_idx)} val | Device: {dev}")

Datos cargados: 1111 train / 329 val | Device: cuda


### 4. Inicialización del Modelo, Pérdida y Optimización

In [5]:
model = ThermalDepartureTimeNet().to(dev)

# Inicializamos SqrtScaledMSELoss con el factor de escala requerido por el modelo mono-modal
crit = SqrtScaledMSELoss(scale=CONFIG["time_scale"])

opt = torch.optim.SGD(model.parameters(), lr=CONFIG["lr"], momentum=CONFIG["momentum"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[30, 60, 80], gamma=0.8)

print(f"Parámetros totales de DSTFS-adapted: {sum(p.numel() for p in model.parameters()):,}")

Parámetros totales de DSTFS-adapted: 11,570,769


### 5. Ciclo de Entrenamiento e Impresión de Métricas Unificadas

In [6]:
best_mae = float("inf")
no_imp = 0

for ep in range(1, CONFIG["epochs"] + 1):
    model.train()
    train_loss, n = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(dev), y.to(dev)
        opt.zero_grad(set_to_none=True)
        loss = crit(model(x), y)
        loss.backward()
        opt.step()
        train_loss += loss.item() * x.size(0)
        n += x.size(0)

    scheduler.step()

    # Evaluación unificada en segundos reales desescalados
    val_m = eval_dstfs_metrics(model, val_loader, dev)
    train_m = eval_dstfs_metrics(model, train_eval_loader, dev)

    v_mae = val_m["mae"]
    is_best = v_mae < best_mae - CONFIG["min_delta"]
    if is_best: 
        best_mae, no_imp = v_mae, 0
        torch.save(model.state_dict(), "../DSTFS_adapted_best.pt")
    else: 
        no_imp += 1
    
    print(f"Ep {ep:03d} | Loss: {train_loss/n:.4f} | "
          f"TrMAE: {train_m['mae']:5.2f}s | ValMAE: {v_mae:5.2f}s | "
          f"RMSE: {val_m['rmse']:5.2f}s | R2: {val_m['r2']:.4f} | MAPE: {val_m['mape']:5.2f}% | "
          f"Acc60: {val_m['acc60']:.2f}% | Acc120: {val_m['acc120']:.2f}% {'*' if is_best else ''}")
    
    if no_imp >= CONFIG["patience"]:
        print(f"Early stop alcanzado. Mejor Val MAE: {best_mae:.2f}s")
        break

Ep 001 | Loss: 5.3128 | TrMAE: 190.18s | ValMAE: 196.06s | RMSE: 255.68s | R2: -1.4050 | MAPE: 100.47% | Acc60: 27.66% | Acc120: 40.43% *
Ep 002 | Loss: 3.6151 | TrMAE: 166.87s | ValMAE: 175.85s | RMSE: 234.31s | R2: -1.0197 | MAPE: 102.88% | Acc60: 30.09% | Acc120: 45.29% *
Ep 003 | Loss: 2.1138 | TrMAE: 137.67s | ValMAE: 145.18s | RMSE: 193.25s | R2: -0.3739 | MAPE: 79.53% | Acc60: 32.52% | Acc120: 52.89% *
Ep 004 | Loss: 1.1971 | TrMAE: 109.26s | ValMAE: 117.53s | RMSE: 159.81s | R2: 0.0605 | MAPE: 67.44% | Acc60: 38.60% | Acc120: 66.26% *
Ep 005 | Loss: 0.7262 | TrMAE: 84.17s | ValMAE: 99.40s | RMSE: 136.53s | R2: 0.3143 | MAPE: 60.78% | Acc60: 43.16% | Acc120: 73.86% *
Ep 006 | Loss: 0.4757 | TrMAE: 70.08s | ValMAE: 82.76s | RMSE: 116.71s | R2: 0.4989 | MAPE: 56.20% | Acc60: 51.67% | Acc120: 77.51% *
Ep 007 | Loss: 0.3629 | TrMAE: 66.17s | ValMAE: 74.88s | RMSE: 104.87s | R2: 0.5954 | MAPE: 56.34% | Acc60: 60.18% | Acc120: 79.64% *
Ep 008 | Loss: 0.2706 | TrMAE: 53.92s | ValMAE: 6